# Руководство по SimulacraBench

Мы воспользуемся учебной схемой `data/sample.json`, чтобы разобраться в
структуре задачи и понять техническую мотивацию конкурса. Формат подачи
решения, правила подсчёта очков и регламент приведены в `README.md`.

In [ ]:
import os
import sys
from pathlib import Path

# Этот блокнот находится в tutorials/, а система проверки — в корне
# репозитория. Все пути ниже (config.yml, data/sample.json, _sandbox/)
# указаны относительно корня репозитория. Поэтому сначала находим корень и
# переходим в него, чтобы блокнот работал одинаково независимо от того,
# откуда запущен Jupyter: из этой папки или уровнем выше.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("запускайте этот блокнот из клона репозитория")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# Те же модули, что использует score.py. Этот блокнот не меняет
# логику проверки: при оценке здесь вызывается тот же код, который будет
# оценивать ваши результаты.
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## 1. Форма задачи

`make_sandbox.py` превращает схему в набор данных той же структуры, что и
настоящий набор данных: с теми же столбцами, вариантами ответов и логикой
переходов. Распределения отдельных переменных и зависимости между ними
вымышлены, поэтому отлаженный здесь конвейер можно будет перенести на
реальные данные, а настроенную здесь модель — нет.

Он записывает `respondents.parquet`, содержащий данные по каждому
респонденту и переменную `role`, которая указывает его назначение, а также
`schema.json` — тот самый файл, который получает ваш `predict()`. Роли
назначаются один раз на этом этапе и записываются на диск. `score.py`
только считывает их; сам он выборку не разбивает.

In [ ]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "виден в обеих фазах",
           "DEV": "оценивается в фазе 1, виден в фазе 2",
           "FINAL": "оценивается в фазе 2"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"число респондентов": counts,
                    "значение": [MEANING[r] for r in counts.index]}).to_string())

### Что объявляет схема

У каждого пункта четыре ключа: `question` — формулировка вопроса, `class` —
класс пункта, `values` — допустимые ответы и `gate` — условие, определяющее,
кому задаётся этот пункт.

Вся задача строится вокруг этих трёх классов. **`GIVEN`** виден всем и
никогда не оценивается. **`PREDICT`** скрыт для респондентов, ответы
которых нужно предсказать, и каждая пустая ячейка оценивается. Поля класса
**`EXCLUDE`** — идентификаторы, ключи записей, свободный текст — вообще не
включаются в таблицу. Поэтому фильтруйте поля по `class`, а не исходите из
того, что
схема и таблица содержат одни и те же столбцы.

Здесь разделение намеренно отражает разницу между дешёвой и дорогой частями
анкеты. `GIVEN` — это информация, которая уже есть в основе выборки, списке
переписи или другом обследовании тех же домохозяйств: где живёт человек,
каков размер домохозяйства, есть ли у кого-либо в домохозяйстве телефон.
`PREDICT` — информация, для
получения которой требуется интервью. На этом различии построена часть 2.

In [ ]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"пункт": name,
                 "класс": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "условие перехода": gate.get("parent", "-"),
                 "задаётся, если": ", ".join(gate.get("observed_if", [])) or "-",
                 "варианты ответа": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

`K` — число возможных значений для пункта, **включая специальное значение
`NA_GATED`**. Для пункта, который задаётся только при выполнении
определённого условия, это означает, что в векторе вероятностей есть на
одну позицию больше, чем обычных вариантов ответа: отдельная позиция
соответствует случаю, когда вопрос респонденту не задавался. Эта позиция
всегда последняя в векторе вероятностей. Значение `K` также используется
для расчёта равномерного эталона `U`.

`would_return` зависит от `clinic_wait`, который, в свою очередь, зависит
от `visited_clinic`: это цепочка из двух уровней. Тому, кто не обращался в
поликлинику, не задавали ни вопрос о времени ожидания, ни вопрос о
повторном визите. Для обоих пунктов правильное значение в этом случае —
`NA_GATED`.

In [ ]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("число респондентов").head(12).to_string())

Читайте эту таблицу как описание правил пропуска вопросов. Если
`visited_clinic` не равен `Yes`, вопросы `clinic_wait` и `would_return`
респонденту не задавались, поэтому их правильное значение — `NA_GATED`.
**Когда условие, определяющее показ вопроса, известно, такие случаи можно
определить заранее.** Для них модель должна отдать вероятность `NA_GATED`.

### Что получает `predict()`

Для текущей фазы `score.py` объединяет видимых и скрытых респондентов,
помещая видимых первыми. Затем у скрытых респондентов он заменяет все
значения `PREDICT` на пустые. `predict()` получает эту таблицу и должен
вернуть по одному вектору вероятностей для каждой такой пустой ячейки.

`NaN` означает только одно: *эта ячейка скрыта, и её нужно предсказать*. Он
никогда не означает, что респондент не ответил на вопрос. Настоящий отказ
от ответа представлен обычным вариантом, например `Prefer not to answer`, и
включён в список допустимых ответов наравне с остальными.

In [ ]:
frame, cells, truth = sample_rows(sample, respondents, config, PHASE, seed=SEED)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("таблица:", frame.shape, " число ячеек для прогноза:", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

Верхние строки соответствуют видимым респондентам: они заполнены полностью,
и их можно использовать для обучения. В нижних строках находятся скрытые
респонденты: для них виден только блок `GIVEN`.

Для каждой пустой ячейки нужно вернуть один вектор вероятностей в
**каноническом порядке**: строки идут сверху вниз, а внутри каждой строки
пункты следуют в порядке ключей `schema["items"]`, а не в порядке
`frame.columns`, который может отличаться.

Элементы каждого вектора должны соответствовать значениям из `values` в том
же порядке. Если пункт задаётся только при выполнении определённого
условия, в конце добавляется отдельная позиция для `NA_GATED`. Порядок
всегда берите из схемы, а не из данных: даже если какой-то вариант ответа
не встретился ни у одного респондента, он всё равно занимает свою позицию в
векторе.

In [6]:
print(pd.DataFrame(cells, columns=["строка", "respondent_id", "пункт"]).head(8)
      .to_string(index=False))

 строка respondent_id                пункт
   2657       R000011       visited_clinic
   2657       R000011          clinic_wait
   2657       R000011         would_return
   2657       R000011 trusts_health_advice
   2658       R000022       visited_clinic
   2658       R000022          clinic_wait
   2658       R000022         would_return
   2658       R000022 trusts_health_advice


### Подсчёт очков

Базовый прогноз по выборке: для каждого скрытого респондента используется
сглаженное распределение ответов по каждому пункту, без учёта его
индивидуальных характеристик. `skill` равен 0 для равномерного прогноза и 1 для
идеального прогноза. Именно по значению `skill` формируется рейтинг
участников.

In [ ]:
def hidden_cells(frame, items):
    '''Все пустые ячейки в том порядке, в котором predict() должен вернуть для них прогнозы.'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("равномерный эталон (при натуральном логарифме)", "%.4f" % result["uniform_reference"]),
       ("логарифмическая оценка", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("skill равен 0 для равномерного прогноза и 1 для идеального прогноза.")

Вот и весь контракт. Отправляемое решение — это файл `main.py` с функцией
`predict()`, которая возвращает такие векторы. `score.py` запускает её так
же, как это будет делать проверяющая программа, а
`tools/check_submission_zip.py` проверяет корректность структуры
загружаемого архива.

---

## 2. Что даёт хорошая модель

Если бенчмарк оценивает, насколько хорошо модель предсказывает ответы
людей, возникает естественный вопрос: не означает ли это, что людей можно
просто перестать спрашивать? Здесь мы покажем, как сочетать алгоритмические
прогнозы с данными, собранными у респондентов.

Нас интересует один показатель по населению: доля домохозяйств, доверяющих
рекомендациям своей местной поликлиники по вопросам здоровья. Дешёвый
блок — регион, городская или сельская местность, размер домохозяйства,
наличие телефона — уже известен для каждого домохозяйства в основе выборки
из административных данных или предыдущего обследования. Дорогой блок
требует личного опроса, а бюджета хватает лишь на несколько сотен интервью.

У вас три варианта.

1. **Только интервью.** Опросить 300 домохозяйств, рассчитать долю и
   построить доверительный интервал. Такой подход корректен, но его
   точность ограничена объёмом выборки в 300 интервью.
2. **Только модель.** Применить модель к дешёвому блоку для каждого
   домохозяйства и опубликовать средний прогноз. Это почти ничего не
   стоит, но результат будет *ошибаться ровно настолько, насколько
   ошибается сама модель* — без доверительного интервала и без возможности
   оценить величину этой ошибки.
3. **И то и другое.** Использовать модель для всех домохозяйств, а затем
   по 300 интервью оценить ошибку модели и скорректировать прогноз. Это
   **prediction-powered inference**, и именно этот подход мы рассмотрим
   далее.

Третий вариант и есть то, что нам нужно: он остаётся корректным независимо
от качества модели и *даёт более точную оценку, когда модель хороша*.

In [ ]:
# Оцениваемая величина: доля тех, кто хотя бы отчасти доверяет рекомендациям по вопросам здоровья.
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# Модель обучена на респондентах прошлых волн — с ролью TRAIN,
# то есть на том же видимом блоке, на котором обучается модель участника.
# Домохозяйства, которые мы собираемся опросить, она никогда не видит.
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(дешёвый блок) для каждого домохозяйства

# Совокупность, для которой мы хотим получить оценку: домохозяйства, не использованные для обучения модели.
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # известно только потому, что данные вымышлены

table([("модель обучена на респондентах прошлых волн:", "%d" % past.sum()),
       ("домохозяйств в оцениваемой совокупности:", "%d" % len(frame_rows)),
       ("корреляция между прогнозом и ответом:", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("истинная доля (которую в реальном обследовании мы не знаем):", "%.3f" % TRUTH)])

Теперь выберем 300 домохозяйств для интервью и рассчитаем все три оценки.

Интервал, основанный только на интервью, — стандартный. Интервал с опорой
на прогноз строится из среднего прогноза модели по домохозяйствам, которые
вы **не** опрашивали, с поправкой на среднюю ошибку модели среди
опрошенных:

```
оценка = среднее(прогноз | не опрошены) - [ среднее(прогноз | опрошены) - среднее(ответ | опрошены) ]
                ↑ модель, применённая ко всем      ↑ измеренная ошибка модели
```

Выражение в скобках и есть основной механизм коррекции. Оно рассчитывается
по реальным ответам, поэтому требует проведения интервью, и устраняет
смещение модели независимо от его величины.

In [ ]:
def estimates(f, interviewed, rest):
    '''Оценки по интервью и с опорой на прогноз, каждая со своей стандартной ошибкой.'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  ширина %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("истинное значение", "%.3f" % TRUTH),
       ("только интервью", band(classical)),
       ("с опорой на прогноз", band(powered)),
       ("только модель (без интервью)", "%.3f  [доверительного интервала нет]"
        % predicted[frame_rows].mean())])

Одна выборка сама по себе ничего не доказывает — интервалу могло просто
повезти. Важно, как метод ведёт себя на множестве повторных обследований:
содержит ли интервал истинное значение примерно в 95 % случаев и насколько
он широк? Повторим весь эксперимент тысячу раз, каждый раз выбирая новые
300 домохозяйств.

In [ ]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # оценка, стандартная ошибка, оценка, стандартная ошибка


def summarize(trials, label):
    rows = []
    for name, point, se in (("только интервью", trials[:, 0], trials[:, 1]),
                            ("с опорой на прогноз", trials[:, 2], trials[:, 3])):
        rows.append({"метод": name,
                     "средняя ширина": (2 * 1.96 * se).mean(),
                     "содержит истинное значение": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "модель, которая хорошо предсказывает")
narrower = 1 - good.loc[1, "средняя ширина"] / good.loc[0, "средняя ширина"]
print("\nинтервал на %.0f %% уже при тех же %d интервью." % (100 * narrower, N_INTERVIEWS))
print("чтобы достичь такой же точности только за счёт интервью, понадобилось бы около %d." % round(N_INTERVIEWS / (1 - narrower) ** 2))

Оба интервала содержат истинное значение примерно в 95 % случаев — именно
этого мы и ожидаем от 95-процентных доверительных интервалов. Интервал с
опорой на прогноз просто **уже** при том же объёме полевой работы.
Последняя строка отражает смысл всего упражнения: хорошая модель не
заменяет интервью, а делает каждое интервью более информативным.

### Что происходит, когда модель плоха

Очевидное возражение: всё это работает только тогда, когда модель хорошо
предсказывает, а риск как раз и состоит в том, чтобы ей довериться.
Рассмотрим ту же процедуру с моделью, обученной на **другой совокупности**.

In [ ]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # другая совокупность
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("корреляция между прогнозом и ответом:", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("только модель (без интервью)", "%.3f   в сравнении с истинным значением %.3f   <- ошибка %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "модель, которая плохо переносится на новые данные")

Обратите внимание: оценка, полученная только с помощью модели, отличается
от истинного значения более чем на одну десятую, и по самому результату вы
никак не смогли бы об этом узнать — нет ни доверительного интервала, ни
предупреждения, только число, которое выглядит столь же убедительно, как
правильная оценка. Замена полевого обследования моделью приводит к
смещению.

Интервал с опорой на прогноз по-прежнему содержит истинное значение
примерно в 95 % случаев. При этом он не уже, чем интервал, основанный
только на интервью: бесполезная модель не даёт выигрыша в точности.
Поправочный член оценивает ошибку модели по 300 реальным интервью и вносит
соответствующую поправку — именно для этого он и нужен.

### Зачем для этого нужен бенчмарк

Ширина доверительного интервала напрямую зависит от качества модели.
Поэтому важно тщательно измерять качество прогнозов на реальных опросных
инструментах с помощью строгого правила подсчёта очков: более высокий
`skill` в таблице результатов означает более узкий доверительный интервал
в реальном исследовании или такую же точность при меньшем числе интервью.